In [0]:
--BRONZE
--Verification customers
   
    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_customers;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_customers GROUP BY _batch_id ORDER BY _batch_id;

--Verification inventory    

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_inventory;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_inventory GROUP BY _batch_id ORDER BY _batch_id;

--Verification order_items

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_order_items;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_order_items GROUP BY _batch_id ORDER BY _batch_id;

--Verification orders

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_orders;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_orders GROUP BY _batch_id ORDER BY _batch_id;

--Verification payments

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_payments;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_payments GROUP BY _batch_id ORDER BY _batch_id;

--Verification products

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_products;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_products GROUP BY _batch_id ORDER BY _batch_id;

--Verification returns

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_returns;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_returns GROUP BY _batch_id ORDER BY _batch_id;

--Verification sellers

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_sellers;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_sellers GROUP BY _batch_id ORDER BY _batch_id;

--Verification shipments

    SELECT COUNT(*) AS total_records FROM ecomstore.ecomlake.bronze_shipments;
    SELECT _batch_id, COUNT(*) AS cnt FROM ecomstore.ecomlake.bronze_shipments GROUP BY _batch_id ORDER BY _batch_id;

--SILVER
--Verification customers

    SELECT is_current, COUNT(*) AS record_count
    FROM ecomstore.ecomlake.silver_customers_scd
    GROUP BY is_current;

    SELECT customer_id, city, customer_tier, start_date, end_date, is_current
    FROM ecomstore.ecomlake.silver_customers_scd
    WHERE customer_id IN (
        SELECT customer_id FROM ecomstore.ecomlake.silver_customers_scd GROUP BY customer_id HAVING COUNT(*) > 1
    )
    ORDER BY customer_id, start_date
    LIMIT 20;

--Verification inventory

    SELECT warehouse_id,
           COUNT(*)                                    AS total_skus,
           SUM(stock_quantity)                         AS total_stock,
           SUM(CASE WHEN is_below_reorder_level THEN 1 ELSE 0 END) AS skus_below_reorder
    FROM ecomstore.ecomlake.silver_inventory
    GROUP BY warehouse_id
    ORDER BY warehouse_id;

    SELECT product_id, warehouse_id, stock_quantity, reorder_level
    FROM ecomstore.ecomlake.silver_inventory
    WHERE is_below_reorder_level = true
    ORDER BY stock_quantity ASC
    LIMIT 10;

--Verification order_items

    SELECT status, COUNT(*) AS cnt, AVG(total_price) AS avg_item_value
    FROM ecomstore.ecomlake.silver_order_items
    GROUP BY status
    ORDER BY cnt DESC;

--Verification orders 
 
   SELECT status, COUNT(*) AS cnt, AVG(final_amount) AS avg_order_value 
   FROM ecomstore.ecomlake.silver_orders 
   GROUP BY status 
   ORDER BY cnt DESC; 

  --Verification payments 

   SELECT payment_status, payment_method, COUNT(*) AS cnt, AVG(amount) AS avg_amount
   FROM ecomstore.ecomlake.silver_payments
   GROUP BY payment_status, payment_method
   ORDER BY cnt DESC;

--Verification products

   SELECT COUNT(*) AS total_records 
   FROM ecomstore.ecomlake.silver_products;
   
   SELECT _batch_id, COUNT(*) AS cnt 
   FROM ecomstore.ecomlake.silver_products 
   GROUP BY _batch_id 
   ORDER BY _batch_id;

--Verification returns

   SELECT return_status, return_reason, COUNT(*) AS cnt, AVG(refund_amount) AS avg_refund
   FROM ecomstore.ecomlake.silver_returns
   GROUP BY return_status, return_reason
   ORDER BY cnt DESC;

--Verification sellers

   SELECT is_current, COUNT(*) AS record_count
   FROM ecomstore.ecomlake.silver_sellers_scd
   GROUP BY is_current;

   SELECT seller_id, seller_name, seller_tier, start_date, end_date, is_current
   FROM ecomstore.ecomlake.silver_sellers_scd
   WHERE seller_id IN (SELECT seller_id FROM ecomstore.ecomlake.silver_sellers_scd GROUP BY seller_id HAVING COUNT(*) > 1)
   ORDER BY seller_id, start_date
   LIMIT 20;

--Verification shipments

   SELECT shipment_status, COUNT(*) AS cnt, SUM(CASE WHEN actual_delivery_date IS NOT NULL THEN 1 ELSE 0 END) AS delivered_count
   FROM ecomstore.ecomlake.silver_shipments
   GROUP BY shipment_status
   ORDER BY cnt DESC;

--GOLD
--Verification customer_lifetime_value
    
    SELECT clv_segment, COUNT(*) AS total_customers, ROUND(AVG(net_revenue),2) AS avg_net_revenue 
    FROM ecomstore.ecomlake.gold_customer_lifetime_value 
    GROUP BY clv_segment 
    ORDER BY avg_net_revenue DESC;

--Verification daily_sales_summary

    SELECT * FROM ecomstore.ecomlake.gold_daily_sales_summary ORDER BY order_date DESC;

--Verification delivery_sla_report

    SELECT
        carrier,
        SUM(total_shipments)    AS total_shipments,
        SUM(delivered_count)    AS delivered,
        SUM(on_time_count)      AS on_time,
        ROUND(SUM(on_time_count) / NULLIF(SUM(delivered_count), 0) * 100, 2) AS on_time_pct,
        ROUND(SUM(sum_days_to_deliver) / NULLIF(SUM(delivered_count), 0), 1)  AS true_avg_transit_days
    FROM ecomstore.ecomlake.gold_delivery_sla_report
    GROUP BY carrier
    ORDER BY on_time_pct DESC;


    SELECT
        delivery_city,
        delivery_state,
        SUM(total_shipments) AS total_shipments,
        ROUND(SUM(late_count) / NULLIF(SUM(delivered_count), 0) * 100, 2) AS late_pct
    FROM ecomstore.ecomlake.gold_delivery_sla_report
    GROUP BY delivery_city, delivery_state
    HAVING SUM(delivered_count) > 10
    ORDER BY late_pct DESC
    LIMIT 10;

--Verification inventory_health 
   
    SELECT 
        warehouse_id,
        category,
        total_unique_skus,
        out_of_stock_skus,
        skus_below_reorder_level,
        stockout_risk_pct
    FROM 
        ecomstore.ecomlake.gold_inventory_health
    WHERE 
        stockout_risk_pct > 20.0 -- Flag if more than 20% of catalog is at risk
        OR out_of_stock_skus > 0
    ORDER BY 
        stockout_risk_pct DESC, 
        warehouse_id ASC;

--Verification product_returns_trend

    SELECT 
        category,
        seller_name,
        SUM(total_units_sold) AS total_sold,
        SUM(total_units_returned) AS total_returned,
        ROUND((SUM(total_units_returned) / NULLIF(SUM(total_units_sold), 0)) * 100, 2) AS avg_return_rate,
        SUM(damaged_returns) AS damaged_count
    FROM ecomstore.ecomlake.gold_product_returns_trend
    GROUP BY category, seller_name
    HAVING total_sold > 10
    ORDER BY avg_return_rate DESC
    LIMIT 10;

--Verification seller_performance
   
    SELECT seller_name, seller_tier, total_gmv, fulfillment_rate_pct, return_rate_pct, on_time_delivery_pct
    FROM ecomstore.ecomlake.gold_seller_performance
    ORDER BY total_gmv DESC
    LIMIT 10;
